# Prepare data — raw datasets → manifests (Kaggle)

Builds the audio-free **manifests** M2 (baselines) and M3 (fine-tune) consume, from
the licensed sources. Audio stays on this machine; only manifests (paths + text +
durations + recording ids) are written to `data/`. Thin driver over the tested
`shono.data` builders — no scoring or model logic here.

Set the `*_ROOT` paths below to your Kaggle dataset mounts, then run top to bottom.

In [ ]:
!pip install -q librosa soundfile
!pip install -q -e /kaggle/working/shono

In [ ]:
from pathlib import Path

from shono.data import (
    LicenseRegistry, audit_leakage, build_manifest,
    from_common_voice_tsv, from_fleurs_tsv,
)

REPO = Path('/kaggle/working/shono')
DATA = REPO / 'data'
registry = LicenseRegistry.load(DATA / 'licenses.json')

# Dataset mounts — set these to your Kaggle inputs.
CV_ROOT = Path('/kaggle/input/common-voice-bn')      # Common Voice bn (from Mozilla Data Collective)
FLEURS_ROOT = Path('/kaggle/input/fleurs-bn')        # FLEURS bn (google/fleurs, bn_in)
SLR53_ROOT = Path('/kaggle/input/openslr53')         # OpenSLR SLR53
MUCS_ROOT = Path('/kaggle/input/mucs-slr104')        # MUCS 2021 / SLR104 code-switch
LOOP_ROOT = Path('/kaggle/input/bengali-loop')       # Bengali-Loop long-form

## Eval manifests (M2 slices)

Common Voice and FLEURS have well-specified indexes, so the package builds them
directly. `client_id` becomes the recording block for the bootstrap CI.

In [ ]:
cv_test = from_common_voice_tsv(
    CV_ROOT / 'test.tsv', split='test', name='cv-bn-test', version='cv-corpus-26.0',
    audio_root=CV_ROOT,
    durations_tsv=(CV_ROOT / 'clip_durations.tsv' if (CV_ROOT / 'clip_durations.tsv').exists() else None),
)
fleurs_test = from_fleurs_tsv(
    FLEURS_ROOT / 'test.tsv', split='test', name='fleurs-bn-test', audio_root=FLEURS_ROOT,
)
for m in (cv_test, fleurs_test):
    m.validate_against(registry)
    m.to_jsonl(DATA / f'{m.name}.manifest.jsonl')
    print(f'{m.name}: {len(m.segments)} segments, {m.total_hours():.1f} h')

## Sources with dataset-specific layouts

OpenSLR SLR53, MUCS, and Bengali-Loop ship their own index formats. Map each to
rows `{id, audio, text, [recording_id, start_s, duration_s]}` and hand them to
`build_manifest` — adjust the readers below to match your actual downloads. A row
without `duration_s` has it read from the audio file (librosa).

In [ ]:
import csv

def slr53_rows(index_tsv, audio_subdir='wavs'):
    # OpenSLR indexes are typically `utt_id<TAB>transcript` with audio at <utt_id>.wav/.flac.
    with open(index_tsv, encoding='utf-8') as f:
        for utt, *rest in csv.reader(f, delimiter='\t'):
            if rest and rest[0].strip():
                yield {'id': utt, 'audio': f'{audio_subdir}/{utt}.wav', 'text': rest[0].strip()}

def mucs_rows(index_tsv, audio_subdir='audio'):
    # MUCS: `utt_id<TAB>transcript`; recording id from the lecture prefix if present.
    with open(index_tsv, encoding='utf-8') as f:
        for utt, *rest in csv.reader(f, delimiter='\t'):
            if rest and rest[0].strip():
                rec = utt.rsplit('_', 1)[0]
                yield {'id': utt, 'audio': f'{audio_subdir}/{utt}.wav', 'text': rest[0].strip(),
                       'recording_id': rec}

def bengali_loop_rows(segments_json):
    # Bengali-Loop publishes per-recording segment annotations (start, end, text).
    import json
    data = json.loads(Path(segments_json).read_text(encoding='utf-8'))
    for rec in data:  # adjust keys to the actual annotation schema
        vid = rec['recording_id']
        for i, seg in enumerate(rec['segments']):
            yield {'id': f'{vid}-{i:04d}', 'audio': rec['audio'], 'text': seg['text'],
                   'recording_id': vid, 'start_s': seg['start'],
                   'duration_s': seg['end'] - seg['start']}

## Train + remaining eval manifests (M3 mix + long-form/CS eval)

In [ ]:
slr53_train = build_manifest(
    name='slr53-train', source='openslr_slr53', split='train', domain='read',
    version='SLR53', rows=slr53_rows(SLR53_ROOT / 'line_index.tsv'), audio_root=SLR53_ROOT,
)
cv_train = from_common_voice_tsv(
    CV_ROOT / 'train.tsv', split='train', name='cv-bn-train', version='cv-corpus-26.0',
    audio_root=CV_ROOT,
)
mucs_cs_test = build_manifest(
    name='mucs-cs-test', source='mucs_slr104', split='test', domain='code-switch',
    version='SLR104', rows=mucs_rows(MUCS_ROOT / 'test' / 'text'), audio_root=MUCS_ROOT / 'test',
)
mucs_cs_train = build_manifest(
    name='mucs-cs-train', source='mucs_slr104', split='train', domain='code-switch',
    version='SLR104', rows=mucs_rows(MUCS_ROOT / 'train' / 'text'), audio_root=MUCS_ROOT / 'train',
)
loop_test = build_manifest(
    name='bengali-loop-sample', source='bengali_loop', split='test', domain='long-form',
    version='arXiv-2602.14291', rows=bengali_loop_rows(LOOP_ROOT / 'test_segments.json'),
    audio_root=LOOP_ROOT,
)
loop_train = build_manifest(
    name='bengali-loop-train', source='bengali_loop', split='train', domain='long-form',
    version='arXiv-2602.14291', rows=bengali_loop_rows(LOOP_ROOT / 'train_segments.json'),
    audio_root=LOOP_ROOT,
)

for m in (slr53_train, cv_train, mucs_cs_test, mucs_cs_train, loop_test, loop_train):
    m.validate_against(registry)
    m.to_jsonl(DATA / f'{m.name}.manifest.jsonl')
    print(f'{m.name}: {len(m.segments)} segments, {m.total_hours():.1f} h')

## Leakage audit — before training

Audit every eval slice against the whole training mix (id / normalized-text / audio
checksum). Investigate any non-clean result before fine-tuning.

In [ ]:
trains = [slr53_train, cv_train, mucs_cs_train, loop_train]
for ev in (cv_test, fleurs_test, mucs_cs_test, loop_test):
    print(audit_leakage(trains, ev).summary())